# Milestone 1 - Baseline BM25 no BioASQ

Este notebook documenta a primeira milestone do projeto **MedIR-RAG**.

Objetivos:

- carregar o dataset BioASQ;
- construir queries, documentos e julgamentos de relevância (`qrels`);
- aplicar tokenização, normalização e remoção de stop-words;
- criar uma baseline lexical com BM25;
- avaliar a ordenação com `NDCG@10`, `MAP` e `MRR`.

A implementação será executada por etapas para tornar visível o percurso dos dados.

## 1. Configuração

Começamos por importar as bibliotecas. Nesta execução experimental usamos o dataset mais pequeno `BioASQ_2013_TaskB/BioASQ-trainingDataset2b.json`, que está dentro desta pasta.

Este dataset é adequado para testar o pipeline rapidamente. Para os resultados finais, devemos indicar no relatório qual dataset foi usado e manter a mesma coleção em todas as comparações.

In [ ]:
from pathlib import Path
import json
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
import pytrec_eval

DATASET_PATH = Path("BioASQ14-2026-Training-14b.json")

print(f"Dataset: {DATASET_PATH.resolve()}")
print(f"Existe: {DATASET_PATH.exists()}")

Dataset: C:\Users\diogu\Documents\1uni\MEI\RIBT\RIBT_Projeto_MedIR_RAG\Milestone_1\BioASQ2-2014-Training-2b.json
Existe: True


## 2. Carregamento e inspeção do dataset

O BioASQ contém perguntas, documentos relevantes e snippets associados a artigos do PubMed.

In [21]:
with DATASET_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

questions = data["questions"]

print(f"Número de registos: {len(questions)}")
print("Campos da primeira pergunta:")
print(questions[0].keys())
print("\nPrimeira pergunta:")
print(questions[0]["body"])
print(f"\nNúmero de snippets: {len(questions[0].get('snippets', []))}")

Número de registos: 310
Campos da primeira pergunta:
dict_keys(['body', 'concepts', 'documents', 'exact_answer', 'id', 'ideal_answer', 'snippets', 'type'])

Primeira pergunta:
Is Rheumatoid Arthritis more common in men or women?

Número de snippets: 16


## 3. Pré-processamento

Nesta baseline vamos usar minúsculas, remoção de pontuação, tokenização e stop-words em inglês. O objetivo é fazer com que palavras como `Cancer` e `cancer` sejam tratadas da mesma forma.

In [22]:
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords")

try:
    word_tokenize("test")
except LookupError:
    nltk.download("punkt")
    nltk.download("punkt_tab")

STOP_WORDS = set(stopwords.words("english"))


def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    return [token for token in tokens if token not in STOP_WORDS and token.strip()]

example_text = questions[0]["body"]
print("Antes:")
print(example_text)
print("\nDepois:")
print(preprocess_text(example_text))

Antes:
Is Rheumatoid Arthritis more common in men or women?

Depois:
['rheumatoid', 'arthritis', 'common', 'men', 'women']


## 4. Construção das queries, corpus e qrels

- `queries`: pergunta usada para pesquisar;
- `corpus`: documentos que o BM25 vai indexar, construídos a partir dos snippets;
- `qrels`: documentos considerados relevantes para cada pergunta.

In [23]:
queries = {}
corpus = {}
qrels = {}

for question in questions:
    question_id = question.get("id")
    question_text = question.get("body")

    if not question_id or not question_text:
        continue

    queries[question_id] = question_text
    qrels[question_id] = {
        document_id: 1
        for document_id in question.get("documents", [])
    }

    for snippet in question.get("snippets", []):
        document_id = snippet.get("document")
        snippet_text = snippet.get("text", "")

        if not document_id or not snippet_text:
            continue

        corpus[document_id] = corpus.get(document_id, "") + " " + snippet_text

print(f"Queries: {len(queries)}")
print(f"Documentos antes da tokenização: {len(corpus)}")
print(f"Julgamentos de relevância: {len(qrels)}")

Queries: 306
Documentos antes da tokenização: 3539
Julgamentos de relevância: 306


## 5. Tokenização do corpus e das queries

O BM25 recebe listas de tokens, não texto bruto. Por isso aplicamos a mesma função de pré-processamento às perguntas e aos snippets.

In [24]:
tokenized_queries = {
    question_id: preprocess_text(question_text)
    for question_id, question_text in queries.items()
}

tokenized_corpus = {
    document_id: preprocess_text(document_text)
    for document_id, document_text in corpus.items()
}

doc_ids = list(tokenized_corpus)
corpus_tokens = [tokenized_corpus[document_id] for document_id in doc_ids]

print(f"Documentos tokenizados: {len(corpus_tokens)}")
print(f"Tokens no primeiro documento: {len(corpus_tokens[0])}")
print(f"Primeiros tokens: {corpus_tokens[0][:20]}")

Documentos tokenizados: 3539
Tokens no primeiro documento: 35
Primeiros tokens: ['results', 'show', 'high', 'prevalence', 'ra', 'lac', 'women', 'ratio', '52', 'women', 'per', 'man', 'ra', 'lac', 'women', 'common', 'presents', 'clinical', 'characteristics', 'differ']


## 6. Indexação BM25

O BM25 calcula uma pontuação de relevância entre uma query e cada documento. Quanto maior a pontuação, mais compatível é o documento com a pergunta.

In [25]:
bm25 = BM25Okapi(corpus_tokens)

print("Índice BM25 criado.")

Índice BM25 criado.


## 7. Primeira pesquisa

Antes de calcular métricas, vamos observar manualmente os documentos mais bem classificados para uma pergunta.

In [26]:
example_query_id = next(iter(tokenized_queries))
example_query = tokenized_queries[example_query_id]
example_scores = bm25.get_scores(example_query)

ranked_indexes = example_scores.argsort()[::-1][:5]

print(f"Query: {queries[example_query_id]}")
print("\nTop 5 documentos:")

for rank, index in enumerate(ranked_indexes, start=1):
    document_id = doc_ids[index]
    print(f"\n{rank}. score={example_scores[index]:.4f}")
    print(f"ID: {document_id}")
    print(corpus[document_id][:400], "...")

Query: Is Rheumatoid Arthritis more common in men or women?

Top 5 documentos:

1. score=17.8232
ID: http://www.ncbi.nlm.nih.gov/pubmed/10555889
 Myopathy and neuropathy in rheumatoid arthritis with rheumatoid arthritis (RA) have clinical or subclinical evidence of peripheral neuropathy or myopathy The study reveals an increased prevalence of neurogenic but not myogenic changes in patients with RA compared with controls ...

2. score=17.7725
ID: http://www.ncbi.nlm.nih.gov/pubmed/23171360
 Minocycline-associated rimmed vacuolar myopathy in a patient with rheumatoid arthritis ...

3. score=17.7725
ID: http://www.ncbi.nlm.nih.gov/pubmed/16418123
 in 244 female and 91 male patients with rheumatoid arthritis. ...

4. score=17.3827
ID: http://www.ncbi.nlm.nih.gov/pubmed/19642078
 Prevalence of risk factors for statin-induced myopathy in rheumatoid arthritis patients ...

5. score=17.0097
ID: http://www.ncbi.nlm.nih.gov/pubmed/15909088
 Occurrence of chloroquine-induced myopathy after low-do

## 8. Avaliação quantitativa

A avaliação usa apenas queries que têm pelo menos um documento relevante presente no corpus indexado.

Além das métricas pedidas inicialmente, calculamos as métricas indicadas no enunciado completo: `NDCG@5`, `NDCG@10`, `MRR@10`, `MAP` e `Recall@100`.

In [27]:
indexed_qrels = {
    question_id: {
        document_id: relevance
        for document_id, relevance in judgments.items()
        if document_id in tokenized_corpus
    }
    for question_id, judgments in qrels.items()
}
indexed_qrels = {
    question_id: judgments
    for question_id, judgments in indexed_qrels.items()
    if judgments
}

metrics = {
    "ndcg_cut_5",
    "ndcg_cut_10",
    "map",
    "recip_rank",
    "recall_100",
}

evaluator = pytrec_eval.RelevanceEvaluator(indexed_qrels, metrics)
mrr_10_evaluator = pytrec_eval.RelevanceEvaluator(
    indexed_qrels, {"recip_rank"}
)
metric_sums = {metric: 0.0 for metric in metrics}
mrr_10_sum = 0.0
number_of_queries = 0

for question_id, tokenized_query in tokenized_queries.items():
    if question_id not in indexed_qrels:
        continue

    scores = bm25.get_scores(tokenized_query)
    full_run = {
        question_id: {
            document_id: float(score)
            for document_id, score in zip(doc_ids, scores)
        }
    }

    result = evaluator.evaluate(full_run)[question_id]
    for metric in metric_sums:
        metric_sums[metric] += result[metric]

    ranked_indexes = scores.argsort()[::-1][:10]
    top_10_run = {
        question_id: {
            doc_ids[index]: float(scores[index])
            for index in ranked_indexes
        }
    }
    mrr_10_result = mrr_10_evaluator.evaluate(top_10_run)[question_id]
    mrr_10_sum += mrr_10_result["recip_rank"]
    number_of_queries += 1

if number_of_queries == 0:
    raise ValueError("Nenhuma query pode ser avaliada.")

results = {
    "NDCG@5": metric_sums["ndcg_cut_5"] / number_of_queries,
    "NDCG@10": metric_sums["ndcg_cut_10"] / number_of_queries,
    "MRR@10": mrr_10_sum / number_of_queries,
    "MAP": metric_sums["map"] / number_of_queries,
    "Recall@100": metric_sums["recall_100"] / number_of_queries,
}

print(f"Queries avaliadas: {number_of_queries}")
for metric_name, value in results.items():
    print(f"{metric_name:10s}: {value:.4f}")

Queries avaliadas: 305
NDCG@5    : 0.6996
NDCG@10   : 0.6647
MRR@10    : 0.8411
MAP       : 0.5749
Recall@100: 0.8203


## 9. Interpretação

Os valores finais devem ser interpretados em conjunto:

- `NDCG@k`: avalia a qualidade da ordenação até à posição `k`;
- `MAP`: mede a precisão média ao longo da lista recuperada;
- `MRR@10`: verifica quão cedo aparece o primeiro documento relevante;
- `Recall@100`: mede quanto dos documentos relevantes foi recuperado nos primeiros 100 resultados.

Estes resultados constituem a baseline lexical que será comparada com os modelos densos na Milestone 2.

## 10. Exportação dos resultados

Para além do output visível no notebook, guardamos as métricas num ficheiro JSON. Assim podemos reutilizá-las no relatório e comparar esta baseline com os modelos da Milestone 2.

In [28]:
results_path = Path("bm25_results_trainingDataset2b.json")

with results_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "dataset": str(DATASET_PATH),
            "queries_evaluated": number_of_queries,
            "metrics": results,
        },
        file,
        indent=2,
    )

print(f"Resultados guardados em: {results_path.resolve()}")

Resultados guardados em: C:\Users\diogu\Documents\1uni\MEI\RIBT\RIBT_Projeto_MedIR_RAG\Milestone_1\bm25_results_trainingDataset2b.json
